# Exp9 — WTE for HalfCheetahVel task distributions

Same pipeline as `exp2.ipynb` (collect SARS → LWE/WTE embeddings → within/between cosine distances + PCA/LDA),
but for the three CheetahVel velocity ranges:

| ID | Range |
|----|--------|
| 1 | Uniform\[2.5, 3.0\] |
| 2 | Uniform\[0.5, 1.0\] |
| 3 | Uniform\[0.0, 3.0\] |

Continuous actions → `Detect(..., one_hot=False)`. Discrete tasks within a dist are sampled velocities (linspace).

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# repo root when run from scripts/; on Colab often /content/hypex
CODE_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
if (CODE_DIR / "hyper").is_dir():
    HYPER_DIR = CODE_DIR / "hyper"
elif (CODE_DIR / "environments" / "mujoco").is_dir():
    HYPER_DIR = CODE_DIR  # already inside hyper/
else:
    raise FileNotFoundError(f"Cannot find hyper/ from {CODE_DIR}")

SCRIPTS_DIR = CODE_DIR / "scripts" if (CODE_DIR / "scripts").is_dir() else HYPER_DIR.parent / "scripts"

for p in (str(HYPER_DIR), str(SCRIPTS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import environments  # registers HalfCheetahVel-TaskDist*-v0
from environments.mujoco.half_cheetah_vel import HalfCheetahVelEnv
from detect import Detect

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

DATA_ROOT = CODE_DIR / "data" / "chevel"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("CODE_DIR", CODE_DIR)
print("HYPER_DIR", HYPER_DIR)
print("DATA_ROOT", DATA_ROOT)

In [ ]:
# Three CheetahVel task distributions (same as registered HalfCheetahVel-TaskDist{k}-v0)
VEL_TASK_DISTS = {
    1: (2.5, 3.0),  # high-speed band
    2: (0.5, 1.0),  # low-speed band
    3: (0.0, 3.0),  # full range
}
N_TASKS_PER_DIST = 8  # discrete velocities sampled inside each Uniform range


def sample_velocities(dist_id, n_tasks=None):
    """Discrete velocity tasks drawn from Uniform[low, high] via linspace (inclusive)."""
    if n_tasks is None:
        n_tasks = N_TASKS_PER_DIST
    lo, hi = VEL_TASK_DISTS[dist_id]
    return [float(v) for v in np.linspace(lo, hi, n_tasks)]

In [ ]:
EP_LEN = 200
N_EPISODES = 25          # rollouts per velocity task
N = EP_LEN * N_EPISODES   # SARS rows kept per velocity (~5000, like exp2)
SEED = 42

OBS_DIM = 20   # HalfCheetahVel hyper obs
ACTION_DIM = 6  # continuous torque dim (not one-hot)


def vel_filename(vel):
    return f"vel_{vel:.4f}.npy"


def collect_tuples(task_dists, seed=SEED):
    """task_dists: {dist_id: list of target velocities}.

    Saves SARS arrays under data/chevel/task_dist{id}/vel_X.XXXX.npy
    Each row: [s (20), a (6), r (1), s' (20)] → shape (N, 47).
    """
    rng = np.random.default_rng(seed)
    for dist_id, velocities in task_dists.items():
        lo, hi = VEL_TASK_DISTS[dist_id]
        out_dir = DATA_ROOT / f"task_dist{dist_id}"
        out_dir.mkdir(parents=True, exist_ok=True)

        env = HalfCheetahVelEnv(max_episode_steps=EP_LEN, vel_low=lo, vel_high=hi)
        for vel in velocities:
            sar_rows = []
            env.reset_task(vel)
            obs, _ = env.reset(seed=int(rng.integers(0, 1_000_000)))

            while len(sar_rows) < N:
                action = rng.uniform(
                    env.action_space.low, env.action_space.high
                ).astype(np.float64)
                next_obs, reward, terminated, truncated, info = env.step(action)
                row = np.concatenate(
                    [np.asarray(obs, dtype=np.float64).ravel(),
                     action.ravel(),
                     np.asarray([reward], dtype=np.float64),
                     np.asarray(next_obs, dtype=np.float64).ravel()]
                )
                sar_rows.append(row)
                obs = next_obs
                if terminated or truncated:
                    env.reset_task(vel)
                    obs, _ = env.reset()

            sar = np.asarray(sar_rows[:N], dtype=np.float64)
            path = out_dir / vel_filename(vel)
            np.save(path, sar)
            print(f"dist {dist_id} vel={vel:.4f} → {path.name} shape={sar.shape}")
        env.close()


def mean_pairwise_wte(dist_id, velocities):
    """Mean pairwise cosine distance of WTE/LWE embeddings within one task dist."""
    out_dir = DATA_ROOT / f"task_dist{dist_id}"
    embeddings = []
    for vel in velocities:
        sar = np.load(out_dir / vel_filename(vel))
        embeddings.append(np.asarray(detect.lwe(sar, ACTION_DIM)).ravel())
    E = np.stack(embeddings)
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    cos_dists = [
        1.0 - float(np.dot(E[a], E[b]))
        for a in range(len(E))
        for b in range(a + 1, len(E))
    ]
    return float(np.mean(cos_dists))

In [ ]:
# Sample discrete velocities inside each Uniform range and collect SARS
task_dists = {dist_id: sample_velocities(dist_id) for dist_id in VEL_TASK_DISTS}
for dist_id, vels in task_dists.items():
    lo, hi = VEL_TASK_DISTS[dist_id]
    print(f"TaskDist{dist_id} Uniform[{lo}, {hi}] → {len(vels)} vels: {[round(v, 3) for v in vels]}")

collect_tuples(task_dists)

In [ ]:
# Visualise velocity ranges + sampled tasks
fig, ax = plt.subplots(figsize=(8, 2.8))
yticks, ylabels = [], []
for i, (dist_id, (lo, hi)) in enumerate(sorted(VEL_TASK_DISTS.items())):
    y = i
    ax.hlines(y, lo, hi, colors=f"C{i}", linewidth=6, alpha=0.35)
    vels = task_dists[dist_id]
    ax.scatter(vels, [y] * len(vels), c=f"C{i}", s=40, zorder=3, label=f"dist {dist_id}")
    yticks.append(y)
    ylabels.append(f"TD{dist_id}\n[{lo}, {hi}]")

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels)
ax.set_xlabel("target velocity")
ax.set_xlim(-0.1, 3.2)
ax.set_title("CheetahVel task distributions (sampled velocities)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Detect / LWE setup (continuous actions → one_hot=False)
REFERENCE_NUM = 100
VIZ = False

detect = Detect(
    reference_num=REFERENCE_NUM,
    input_dim=OBS_DIM,
    action_dim=ACTION_DIM,
    num_samples=N,
    sars_flag=True,
    one_hot=False,
)
detect.set_reference(OBS_DIM, REFERENCE_NUM, ACTION_DIM)
print("reference shape", tuple(detect.ref.shape), "(expect N_ref × (20+6+1+20)=47)")

In [ ]:
# Mean embedding per task distribution + between-dist cosine distances
mean_embeddings = []
dist_ids = []
for dist_id, velocities in task_dists.items():
    out_dir = DATA_ROOT / f"task_dist{dist_id}"
    embeddings = []
    for vel in velocities:
        sar = np.load(out_dir / vel_filename(vel))
        embedding = detect.lwe(sar, ACTION_DIM)
        embeddings.append(np.asarray(embedding).ravel())
        if VIZ:
            print(f"--- dist {dist_id} vel={vel:.4f} sar={sar.shape} emb[:5]={embedding[:5]} ---")

    mean_embeddings.append(np.mean(np.stack(embeddings), axis=0))
    dist_ids.append(dist_id)

V = np.stack(mean_embeddings)
V = V / np.linalg.norm(V, axis=1, keepdims=True)
D = 1.0 - (V @ V.T)
df_mean = (
    pd.DataFrame(D, index=dist_ids, columns=dist_ids)
    .rename_axis("dist")
    .reset_index()
)
print("cosine distances between mean embeddings:")
display(df_mean.round(4))

In [ ]:
# Within-distribution mean pairwise WTE (cosine distance)
mean_dists = []
dist_ids_within = []
for dist_id, velocities in task_dists.items():
    d = mean_pairwise_wte(dist_id, velocities)
    mean_dists.append(d)
    dist_ids_within.append(dist_id)

df_within = pd.DataFrame({"dist": dist_ids_within, "d": mean_dists})
print("mean pairwise cosine distances within each task distribution:")
display(df_within.round(4))

In [ ]:
# PCA of per-velocity task embeddings
rows, labels, vels = [], [], []
for dist_id, velocities in task_dists.items():
    out_dir = DATA_ROOT / f"task_dist{dist_id}"
    for vel in velocities:
        sar = np.load(out_dir / vel_filename(vel))
        emb = np.asarray(detect.lwe(sar, ACTION_DIM)).ravel()
        rows.append(emb)
        labels.append(dist_id)
        vels.append(vel)

X = np.stack(rows)
labels = np.asarray(labels)
X_n = X / np.linalg.norm(X, axis=1, keepdims=True)

Z = PCA(n_components=2, random_state=0).fit_transform(X_n)

fig, ax = plt.subplots(figsize=(10, 8))
for dist_id in sorted(set(labels.tolist())):
    mask = labels == dist_id
    sc = ax.scatter(Z[mask, 0], Z[mask, 1], s=60, label=f"dist {dist_id}")
    color = sc.get_facecolors()[0]
    ax.scatter(
        Z[mask, 0].mean(), Z[mask, 1].mean(),
        s=160, marker="x", linewidths=2, c=[color],
    )
for (x, y), v, d in zip(Z, vels, labels):
    ax.annotate(f"{d}:{v:.2f}", (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_title("PCA — CheetahVel WTE embeddings")
ax.legend(fontsize=8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
fig.tight_layout()
plt.show()

In [ ]:
# LDA of per-velocity task embeddings
n_classes = len(np.unique(labels))
n_components = min(2, n_classes - 1)
lda = LinearDiscriminantAnalysis(
    n_components=n_components, solver="eigen", shrinkage="auto"
)
Z_lda = lda.fit_transform(X_n, labels)

fig, ax = plt.subplots(figsize=(10, 8))
for dist_id in sorted(set(labels.tolist())):
    mask = labels == dist_id
    if Z_lda.shape[1] == 1:
        xs = Z_lda[mask, 0]
        ys = np.zeros_like(xs) + 0.02 * (dist_id - 2)
        sc = ax.scatter(xs, ys, s=60, label=f"dist {dist_id}")
        color = sc.get_facecolors()[0]
        ax.scatter(xs.mean(), ys.mean(), s=160, marker="x", linewidths=2, c=[color])
    else:
        sc = ax.scatter(Z_lda[mask, 0], Z_lda[mask, 1], s=60, label=f"dist {dist_id}")
        color = sc.get_facecolors()[0]
        ax.scatter(
            Z_lda[mask, 0].mean(), Z_lda[mask, 1].mean(),
            s=160, marker="x", linewidths=2, c=[color],
        )
ax.set_title("LDA — CheetahVel WTE embeddings")
ax.legend(fontsize=8)
ax.set_xlabel("LD1")
ax.set_ylabel("LD2" if Z_lda.shape[1] > 1 else "")
fig.tight_layout()
plt.show()